# DVC: Data Version Control for ML

## What Is DVC?

Git is great for tracking code — but what about your 50GB training dataset?  
Git would refuse to store it (file too large), and even if it could, you'd fill up everyone's hard drive.

**DVC** (Data Version Control) solves this by being a companion to Git:
- Git tracks your **code** and small config files
- DVC tracks your **large files** (datasets, models) and where they live
- DVC stores a tiny **pointer file** (`.dvc`) in Git — the actual data goes to S3/GCS/SSH/local storage

**Analogy**: Git is like a library card catalog. DVC is like the actual books stored in a warehouse.  
The catalog tells you where each book is; the warehouse has the actual content.

## Resources

- **Docs**: [https://dvc.org/doc](https://dvc.org/doc)
- **GitHub**: [https://github.com/iterative/dvc](https://github.com/iterative/dvc)
- **YouTube — DVC tutorial**: [https://www.youtube.com/watch?v=kLKBcPonMYw](https://www.youtube.com/watch?v=kLKBcPonMYw)
- **YouTube — DVC pipelines**: [https://www.youtube.com/watch?v=71IGzyH95UY](https://www.youtube.com/watch?v=71IGzyH95UY)

## Installation

```bash
pip install dvc
pip install dvc-s3     # if using S3 as remote storage
pip install dvc-gdrive # if using Google Drive
pip install dvc-ssh    # if using SSH server
```

Initialize DVC in a Git repo:
```bash
git init
dvc init
git add .dvc .dvcignore
git commit -m 'Initialize DVC'
```

In [ ]:
import os, subprocess, tempfile, shutil, json
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score
import pickle

try:
    import dvc.api
    DVC_AVAILABLE = True
    result = subprocess.run(['dvc', '--version'], capture_output=True, text=True)
    print(f"DVC version: {result.stdout.strip()}")
except ImportError:
    DVC_AVAILABLE = False
    print("DVC not installed — simulated output shown. Install: pip install dvc")

# Create a temporary project directory for demos
PROJECT_DIR = tempfile.mkdtemp(prefix='dvc_demo_')
print(f"Demo project directory: {PROJECT_DIR}")

## Core Concept 1: How DVC Tracks Files

When you run `dvc add mydata.csv`, DVC:
1. Computes the MD5 hash of the file
2. Copies the file to `.dvc/cache/` (organized by hash)
3. Creates `mydata.csv.dvc` — a small YAML file containing the hash and size
4. Adds `mydata.csv` to `.gitignore` (so Git doesn't try to track the large file)

You then `git add mydata.csv.dvc` and commit the pointer.  
The actual data is in the cache or a remote storage.

### What a .dvc file looks like:

In [ ]:
# Show what a .dvc pointer file looks like
example_dvc_file = {
    "outs": [
        {
            "md5": "a2b4c6d8e0f1234567890abcdef12345",
            "size": 52428800,  # 50 MB
            "path": "data/train.csv",
            "isexec": False,
        }
    ]
}

import yaml
print("Example data/train.csv.dvc file (what Git tracks):")
print("-" * 50)
# Print as YAML (DVC format)
for key, value in example_dvc_file.items():
    print(f"{key}:")
    for item in value:
        for k, v in item.items():
            print(f"  - {k}: {v}")
print("-" * 50)
print()
print("Key insight:")
print("  - This tiny file (< 1KB) goes into Git")
print("  - The actual 50MB CSV goes to DVC remote (S3, GCS, etc.)")
print("  - MD5 hash uniquely identifies the exact version of the data")
print("  - Different hash = different data version")

# Simulate DVC commands
commands = [
    ("dvc add data/train.csv",
     "Creates data/train.csv.dvc + adds data/train.csv to .gitignore"),
    ("git add data/train.csv.dvc",
     "Stage the pointer file for commit"),
    ("git commit -m 'Add training data v1'",
     "Commit the pointer (not the data)"),
    ("dvc push",
     "Upload actual data to remote storage (S3/GCS/etc.)"),
    ("dvc pull",
     "Download data from remote to local cache"),
    ("dvc checkout",
     "Restore files from cache to working directory"),
]

print("\nDVC workflow commands:")
for cmd, description in commands:
    print(f"  $ {cmd}")
    print(f"    → {description}")

## Core Concept 2: DVC Remotes — Where Data Lives

A **remote** is external storage for your tracked files.  
Common remotes:
- AWS S3: `s3://my-bucket/dvc-storage`
- Google Cloud Storage: `gs://my-bucket/dvc-storage`
- Azure Blob: `azure://my-container/dvc-storage`
- SSH server: `ssh://user@host/path/to/storage`
- Local path: `/mnt/shared-drive/dvc-storage` (for shared filesystems)

In [ ]:
print("Setting up DVC remotes (terminal commands):")
print()

remote_examples = [
    ("Local storage (for demos)",
     "dvc remote add -d localremote /tmp/dvc-remote"),
    ("AWS S3",
     "dvc remote add -d s3remote s3://my-company-bucket/ml-data"),
    ("Google Cloud Storage",
     "dvc remote add -d gcsremote gs://my-gcs-bucket/ml-data"),
    ("Azure Blob Storage",
     "dvc remote add -d azureremote azure://mycontainer/ml-data"),
    ("SSH server",
     "dvc remote add -d sshremote ssh://user@ml-server.com/home/user/dvc-data"),
]

for name, cmd in remote_examples:
    print(f"  {name}:")
    print(f"    $ {cmd}")
    print()

print("After adding: settings saved to .dvc/config (committed to Git)")
print()

# Simulate local remote for demo
LOCAL_REMOTE = os.path.join(tempfile.gettempdir(), 'dvc_demo_remote')
os.makedirs(LOCAL_REMOTE, exist_ok=True)

print(f"Demo: using local directory as DVC remote: {LOCAL_REMOTE}")

if DVC_AVAILABLE:
    os.chdir(PROJECT_DIR)
    subprocess.run(['git', 'init'], capture_output=True)
    subprocess.run(['dvc', 'init'], capture_output=True)
    subprocess.run(['dvc', 'remote', 'add', '-d', 'localremote', LOCAL_REMOTE], capture_output=True)
    print("DVC initialized and local remote configured")
else:
    print("[Simulated] DVC initialized, local remote configured")

## Core Concept 3: DVC Pipelines

DVC **pipelines** define your ML workflow as a series of stages with explicit dependencies and outputs.  
This makes your experiments **reproducible** — anyone can run `dvc repro` to recreate your results.

```yaml
# dvc.yaml
stages:
  prepare:
    cmd: python prepare.py
    deps:
      - prepare.py
      - data/raw.csv
    outs:
      - data/processed.csv

  train:
    cmd: python train.py
    deps:
      - train.py
      - data/processed.csv
    outs:
      - models/model.pkl
    metrics:
      - metrics.json: {cache: false}
    params:
      - params.yaml:
        - model.n_estimators
        - model.max_depth
```

`dvc repro` checks what changed and only reruns what's needed — like `make` for ML.

In [ ]:
# Simulate a complete DVC pipeline with all required files

# 1. Create synthetic raw data
np.random.seed(42)
X, y = make_classification(n_samples=2000, n_features=20, random_state=42)
raw_df = pd.DataFrame(X, columns=[f'f{i}' for i in range(20)])
raw_df['label'] = y

raw_path = os.path.join(PROJECT_DIR, 'data_raw.csv')
processed_path = os.path.join(PROJECT_DIR, 'data_processed.csv')
model_path = os.path.join(PROJECT_DIR, 'model.pkl')
metrics_path = os.path.join(PROJECT_DIR, 'metrics.json')

raw_df.to_csv(raw_path, index=False)
print(f"Raw data: {len(raw_df)} rows saved")


def run_prepare_stage(raw_path, processed_path):
    """Mirrors what prepare.py would do."""
    df = pd.read_csv(raw_path)
    df = df.dropna()
    # Feature engineering
    df['f_sum'] = df[[f'f{i}' for i in range(10)]].sum(axis=1)
    df.to_csv(processed_path, index=False)
    print(f"  prepare: {len(df)} rows, {len(df.columns)} cols → {processed_path}")


def run_train_stage(processed_path, model_path, metrics_path, params):
    """Mirrors what train.py would do."""
    df = pd.read_csv(processed_path)
    X = df.drop('label', axis=1).values
    y = df['label'].values
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    model = RandomForestClassifier(
        n_estimators=params['n_estimators'],
        max_depth=params['max_depth'],
        random_state=42
    )
    model.fit(X_train, y_train)

    acc = accuracy_score(y_test, model.predict(X_test))
    auc = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])

    with open(model_path, 'wb') as f:
        pickle.dump(model, f)
    with open(metrics_path, 'w') as f:
        json.dump({'accuracy': acc, 'roc_auc': auc}, f, indent=2)

    print(f"  train:   accuracy={acc:.4f}  roc_auc={auc:.4f} → {model_path}")


params = {'n_estimators': 100, 'max_depth': 8}

print("\nRunning DVC pipeline stages:")
run_prepare_stage(raw_path, processed_path)
run_train_stage(processed_path, model_path, metrics_path, params)

with open(metrics_path) as f:
    metrics = json.load(f)
print(f"\nFinal metrics: {metrics}")

In [ ]:
# Show the dvc.yaml that would define this pipeline
params_yaml = {
    'model': {
        'n_estimators': 100,
        'max_depth': 8
    }
}

dvc_yaml = '''
# dvc.yaml — defines the ML pipeline
stages:
  prepare:
    cmd: python src/prepare.py --input data/raw.csv --output data/processed.csv
    deps:
      - src/prepare.py
      - data/raw.csv         # if this changes, stage reruns
    outs:
      - data/processed.csv   # cached by DVC

  train:
    cmd: python src/train.py
    deps:
      - src/train.py
      - data/processed.csv   # depends on prepare stage output
    outs:
      - models/model.pkl
    metrics:
      - metrics.json:         # tracked but not cached (always regenerated)
          cache: false
    params:
      - params.yaml:          # hyperparameters tracked here
          - model.n_estimators
          - model.max_depth

  evaluate:
    cmd: python src/evaluate.py
    deps:
      - src/evaluate.py
      - models/model.pkl
      - data/test.csv
    metrics:
      - test_metrics.json:
          cache: false
'''

print("dvc.yaml (the pipeline definition):")
print(dvc_yaml)

print("params.yaml (hyperparameters tracked by DVC):")
print(json.dumps(params_yaml, indent=2))

print("\nDVC pipeline commands:")
cmds = [
    ("dvc repro",              "Run all stages whose deps changed"),
    ("dvc repro train",        "Run only the train stage and its deps"),
    ("dvc dag",                "Show pipeline dependency graph"),
    ("dvc metrics show",       "Show metrics.json values"),
    ("dvc params diff",        "Compare params across git commits"),
    ("dvc metrics diff",       "Compare metrics across git commits"),
    ("dvc plots show",         "Visualize metrics history"),
    ("dvc exp run",            "Run experiment with param overrides"),
    ("dvc exp show",           "Table of all experiments"),
]
for cmd, desc in cmds:
    print(f"  $ {cmd:<30s} → {desc}")

## Core Concept 4: DVC Experiments

DVC Experiments let you run and compare many hyperparameter configurations — like W&B Sweeps but Git-native.

In [ ]:
# Simulate DVC experiment tracking
print("DVC Experiments workflow:")
print()
print("# Run experiment with different hyperparameters")
print("$ dvc exp run -n exp-v1 --set-param model.n_estimators=50")
print("$ dvc exp run -n exp-v2 --set-param model.n_estimators=200")
print("$ dvc exp run -n exp-v3 --set-param 'model.n_estimators=100,model.max_depth=5'")
print()
print("# Compare all experiments")
print("$ dvc exp show")
print()

# Simulate the exp show table
exp_data = [
    {"experiment": "baseline",  "n_estimators": 100, "max_depth": 8, "accuracy": 0.8875, "roc_auc": 0.9521},
    {"experiment": "exp-v1",    "n_estimators": 50,  "max_depth": 8, "accuracy": 0.8750, "roc_auc": 0.9433},
    {"experiment": "exp-v2",    "n_estimators": 200, "max_depth": 8, "accuracy": 0.8925, "roc_auc": 0.9587},
    {"experiment": "exp-v3",    "n_estimators": 100, "max_depth": 5, "accuracy": 0.8800, "roc_auc": 0.9498},
]
exp_df = pd.DataFrame(exp_data)
print("Simulated `dvc exp show` output:")
print(exp_df.to_string(index=False))

best = exp_df.loc[exp_df['roc_auc'].idxmax()]
print(f"\nBest experiment: {best['experiment']} (AUC={best['roc_auc']:.4f})")
print()
print("# Apply best experiment as new commit")
print(f"$ dvc exp apply {best['experiment']}")
print("$ git add params.yaml metrics.json")
print("$ git commit -m 'Use best experiment: n_estimators=200'")

In [ ]:
# Using dvc.api to access versioned data in Python
print("dvc.api — access versioned data from Python:")
print()

if DVC_AVAILABLE:
    # This reads data tracked by DVC (requires dvc remote to be configured)
    # with dvc.api.open('data/train.csv', rev='v1.0') as f:
    #     df = pd.read_csv(f)
    print("dvc.api example (requires DVC repo and remote):")
else:
    pass

print("""
import dvc.api

# Open data at a specific git revision
with dvc.api.open('data/train.csv', rev='v2.1.0') as f:
    df = pd.read_csv(f)

# Get data as bytes
data_bytes = dvc.api.read('data/features.pkl', rev='main')
model = pickle.loads(data_bytes)

# Get URL of the data in remote storage (for passing to Spark, etc.)
url = dvc.api.get_url('data/train.parquet')
# → s3://my-bucket/dvc/ab/cd1234...  (the actual S3 path)
""")

print("Key benefit: always get the EXACT data that was used in a given experiment,")
print("regardless of what's currently in data/ folder.")

## Common Pitfalls

| Pitfall | Symptom | Fix |
|---------|---------|-----|
| Committing data to Git | Repo bloated, Git slow | Run `dvc add` before `git add`; check `.gitignore` |
| Forgetting `dvc push` | Teammates can't pull data | Add `dvc push` to CI pipeline after commits |
| Lost cache | `dvc checkout` fails | Run `dvc fetch` to get cache from remote |
| Large dvc.lock conflicts | Hard to merge | Don't manually edit `dvc.lock`; run `dvc repro` after resolving |
| Tracking files inside tracked directory | Double tracking | Use `dvc add` on directories, not individual files inside tracked dirs |
| Stale cache | Old file served | Run `dvc status` to check; `dvc checkout` to restore |
| params.yaml not in dvc.yaml | Param changes don't trigger repro | Always list params in stage `params:` field |

## Mini Project: Full Data Versioning Workflow

In [ ]:
# Simulate the complete DVC workflow for a team

print("Complete DVC team workflow:")
print("=" * 60)

workflow_steps = [
    ("1. Setup (one time)", [
        "git init && dvc init",
        "dvc remote add -d s3remote s3://company-ml-data/dvc",
        "git add .dvc/ && git commit -m 'Init DVC'",
    ]),
    ("2. Add raw data (data engineer)", [
        "dvc add data/raw/customers_2024_q1.csv",
        "git add data/raw/customers_2024_q1.csv.dvc data/.gitignore",
        "git commit -m 'Add Q1 2024 customer data'",
        "dvc push  # upload to S3",
    ]),
    ("3. ML engineer trains (after git pull)", [
        "git pull origin main",
        "dvc pull  # download data from S3",
        "dvc repro  # run pipeline with current params",
        "dvc exp run -n high-lr --set-param model.learning_rate=0.1",
        "dvc exp show  # compare experiments",
    ]),
    ("4. Promote best experiment", [
        "dvc exp apply best-experiment",
        "git add params.yaml dvc.lock metrics.json",
        "git commit -m 'Promote best model: AUC=0.962'",
        "dvc push  # push new model artifact",
        "git push",
    ]),
    ("5. Reproduce any past result", [
        "git checkout v1.2.0  # go to past commit",
        "dvc checkout  # restore data/model from that commit",
        "dvc repro  # reproduce exactly",
    ]),
]

for phase, cmds in workflow_steps:
    print(f"\n{phase}:")
    for cmd in cmds:
        print(f"  $ {cmd}")

print()
print("Result: full reproducibility — any past experiment can be exactly reproduced")
print("by checking out the git commit and running dvc checkout + dvc repro")

## Interview Questions and Answers

In [ ]:
qa = [
    {"q": "How does DVC differ from Git LFS?",
     "a": """Both handle large files that Git can't store, but differently:

Git LFS (Large File Storage):
- Stores large files on a Git LFS server (GitHub, GitLab hosted)
- Transparent: git add/commit/push works normally
- Limited storage on cloud providers (cost scales quickly)
- No pipeline concept
- No experiment tracking

DVC:
- Agnostic to storage: S3, GCS, Azure, SSH, local — anything
- Pipeline DAGs: dvc.yaml defines reproducible ML workflows
- Experiment tracking: compare metrics across runs
- Params tracking: hyperparameter management
- More powerful but more to set up

Choose Git LFS: simple files, small team, GitHub already used
Choose DVC: ML teams, large datasets, reproducibility requirements, pipeline management"""},

    {"q": "What does `dvc repro` do and when does it skip stages?",
     "a": """dvc repro runs the pipeline defined in dvc.yaml. For each stage, it checks:
1. Did any deps (dependency files) change? (MD5 comparison)
2. Did any params change? (tracked in params.yaml)
3. Did the command change?
4. Do the outputs already exist in cache?

If nothing changed → stage is SKIPPED (cache hit)
If anything changed → stage RERUNS and outputs are recached

Example:
- You change params.yaml: model.n_estimators from 100 to 200
- dvc repro: 'prepare' stage is SKIPPED (deps unchanged)
             'train' stage RERUNS (params changed)
             'evaluate' stage RERUNS (model.pkl changed)

This is like 'make' for ML: only rebuild what's necessary."""},

    {"q": "How do you share data with a teammate using DVC?",
     "a": """Steps:
1. Data engineer adds data and pushes:
   dvc add data/train.csv
   git add data/train.csv.dvc && git commit -m 'Add training data'
   dvc push  # uploads to S3/GCS/etc.

2. Teammate receives:
   git pull  # gets the .dvc pointer file
   dvc pull  # downloads actual data from remote

The .dvc pointer file in git tells DVC exactly which version to fetch.
Even if the remote has 100 versions, the teammate always gets the right one.

Requirements:
- Both must have access to the DVC remote (S3 IAM role, GCS service account, etc.)
- DVC remote configured: dvc remote add (committed to .dvc/config in git)
- Credentials configured separately (never in .dvc/config!)"""},

    {"q": "What is the relationship between DVC and MLflow/W&B?",
     "a": """They are complementary, not competing:

DVC handles:
- Data versioning (large files: CSVs, images, audio, model weights)
- Reproducible pipelines (dvc.yaml)
- Git-native experiment tracking (dvc exp)
- Code + data + model linkage through git commits

MLflow/W&B handle:
- Experiment tracking with rich UI
- Real-time metrics during training
- Model registry and serving
- Collaboration and sharing

Many teams use DVC + MLflow together:
- DVC: version the datasets and models on S3, define the pipeline
- MLflow: track experiment metrics, register models for production
- Git: tie everything together (DVC pointers + MLflow run IDs as metadata)"""},
]

for i, item in enumerate(qa, 1):
    print(f"Q{i}: {item['q']}")
    print(f"A:  {item['a'].strip()}")
    print("-" * 65)
    print()

## Summary

| Command | What It Does |
|---------|-------------|
| `dvc init` | Initialize DVC in a git repo |
| `dvc add <file>` | Start tracking a large file |
| `dvc remote add` | Configure external storage |
| `dvc push` | Upload tracked files to remote |
| `dvc pull` | Download tracked files from remote |
| `dvc repro` | Run pipeline, skipping unchanged stages |
| `dvc exp run` | Run experiment with param overrides |
| `dvc exp show` | Compare all experiments in a table |
| `dvc metrics show` | Display tracked metrics |
| `dvc params diff` | Show param changes between git commits |

### Next Steps
1. **Get started**: [https://dvc.org/doc/start](https://dvc.org/doc/start)
2. **DVC + MLflow tutorial**: [https://dvc.org/doc/use-cases/experiment-tracking](https://dvc.org/doc/use-cases/experiment-tracking)
3. **Next**: Learn Kubeflow for running these pipelines at scale on Kubernetes